In [1]:
# Everything this notebook needs. Safe to re-run.
%pip -q install ollama faiss-cpu numpy pandas tqdm requests truststore ipython-autotime flask


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# autotime prints the wall-clock time under every cell from here on.
# Useful here: you can see exactly which steps are expensive.
%load_ext autotime

import textwrap


def pretty_print(*args):
    """print(), but wrapped to 80 columns so long model output stays readable."""
    text = " ".join(str(a) for a in args)
    print(textwrap.fill(text, width=80))

time: 266 µs (started: 2026-08-29 21:44:52 +05:30)


In [3]:
import ollama

client = ollama.Client(host="http://localhost:11434", trust_env=False)

# Which models are actually available? If this errors, Ollama isn't running.
available = [m["model"] for m in client.list()["models"]]
pretty_print("Installed:", ", ".join(sorted(available)))

for needed in ["embeddinggemma:latest", "llama3.1:8b"]:
    mark = "OK  " if needed in available else "MISSING -> ollama pull"
    print(f"  {mark} {needed}")

Installed: embeddinggemma:latest, gemma3:1b, gemma3:27b, gemma3:4b, gemma3n:e4b,
gpt-oss:120b, gpt-oss:20b, llama3.1:8b, nomic-embed-text:latest,
qwen3-embedding:8b, qwen3.5:9b, tinyllama:latest
  OK   embeddinggemma:latest
  OK   llama3.1:8b
time: 334 ms (started: 2026-08-29 21:45:36 +05:30)


In [4]:
import numpy as np

EMBED_MODEL = "embeddinggemma:latest"

resp = client.embeddings(model=EMBED_MODEL, prompt="The quick brown fox jumps over the lazy dog.")
vec = np.asarray(resp["embedding"], dtype="float32")

pretty_print("Model:    ", EMBED_MODEL)
pretty_print("Dimension:", vec.shape)             # 768 for embeddinggemma
pretty_print("First 8:  ", np.round(vec[:8], 4).tolist())
pretty_print("L2 norm:  ", round(float(np.linalg.norm(vec)), 6))

Model:     embeddinggemma:latest
Dimension: (768,)
First 8:   [-0.11029999703168869, 0.05389999970793724, 0.06880000233650208,
-0.022299999371170998, -0.08060000091791153, 0.00419999985024333,
0.03519999980926514, 0.051600001752376556]
L2 norm:   1.0
time: 638 ms (started: 2026-08-29 21:46:56 +05:30)


In [5]:
# Does "similar meaning" really mean "close vector"? Let's check.

def embed_text(text: str) -> np.ndarray:
    """Embed one string and scale it to unit length, so dot product == cosine."""
    v = np.asarray(
        client.embeddings(model=EMBED_MODEL, prompt=text)["embedding"],
        dtype="float32",
    )
    return v / (np.linalg.norm(v) + 1e-12)


a = embed_text("A dog chases a cat in the garden.")
b = embed_text("In the yard, a puppy is running after a kitten.")
c = embed_text("The Fourier transform decomposes a signal into frequencies.")

print(f"cos(a, b)  same idea, no shared words = {float(a @ b):.3f}")
print(f"cos(a, c)  unrelated                   = {float(a @ c):.3f}")

cos(a, b)  same idea, no shared words = 0.787
cos(a, c)  unrelated                   = 0.217
time: 205 ms (started: 2026-08-29 21:48:14 +05:30)


In [6]:
# ── Configuration — everything tunable lives here ────────────────────────
EMBED_MODEL     = "embeddinggemma:latest"
GEN_MODEL       = "llama3.1:8b"      # Part 8 swaps this out; one line to change

WORDS_PER_CHUNK = 300
OVERLAP_WORDS   = 60
TOPK            = 5

CORPUS_DIR    = "corpus_jupyter"
ARTIFACTS_DIR = "rag_artifacts"

GUTENBERG_BOOKS = {
    "Moby-Dick":                     "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice":           "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein":                  "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula":                       "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities":          "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby":              "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace":                 "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre":                     "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray":    "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment":          "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights":             "https://www.gutenberg.org/files/768/768-0.txt",
}

time: 734 µs (started: 2026-08-29 21:50:25 +05:30)


In [7]:
import re
from pathlib import Path

import pandas as pd
import requests
import truststore
from tqdm import tqdm

truststore.inject_into_ssl()   # use the OS certificate store (corporate MITM proxies)

# Gutenberg wraps each book in a licence header and footer. Strip them.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

Path(CORPUS_DIR).mkdir(parents=True, exist_ok=True)

docs = []
for title, url in GUTENBERG_BOOKS.items():
    out_path = Path(CORPUS_DIR) / f"{title.replace(' ', '_')}.txt"

    if not out_path.exists():
        try:
            raw = requests.get(url, timeout=60)
            raw.raise_for_status()
            text = raw.text
            start, end = START_MARK.search(text), END_MARK.search(text)
            if start and end and end.start() > start.end():
                text = text[start.end():end.start()]
            out_path.write_text(text.strip(), encoding="utf-8")
            print(f"  downloaded  {title}")
        except Exception as e:
            print(f"  FAILED      {title}: {e}")
            continue

    docs.append({
        "title": title,
        "text":  out_path.read_text(encoding="utf-8", errors="ignore"),
        "path":  str(out_path),
    })

print(f"\n{len(docs)} books loaded")

  downloaded  Moby-Dick
  downloaded  Pride and Prejudice
  downloaded  Frankenstein
  downloaded  Alice in Wonderland
  downloaded  Dracula
  downloaded  A Tale of Two Cities
  downloaded  The Great Gatsby
  downloaded  Adventures of Sherlock Holmes
  downloaded  War and Peace
  downloaded  Jane Eyre
  downloaded  The Picture of Dorian Gray
  downloaded  Crime and Punishment
  downloaded  Wuthering Heights

13 books loaded
time: 24.3 s (started: 2026-08-29 21:51:04 +05:30)


In [8]:
pd.DataFrame([
    {"title": d["title"], "words": len(d["text"].split())}
    for d in docs
]).sort_values("words", ascending=False).reset_index(drop=True)

,title,words
0,War and Peace,563286
1,Moby-Dick,212796
2,Crime and Punishment,203505
3,Jane Eyre,185390
4,Dracula,161321
5,A Tale of Two Cities,135886
6,Pride and Prejudice,127360
7,Wuthering Heights,115945
8,Adventures of Sherlock Holmes,107562
9,The Picture of Dorian Gray,78979


time: 72.2 ms (started: 2026-08-29 21:51:54 +05:30)


In [9]:
chunks = []          # each: id, title, text, preview, source_path, chunk_index
step = WORDS_PER_CHUNK - OVERLAP_WORDS

for d in docs:
    words = d["text"].split()
    chunk_index = 0
    for i in range(0, len(words), step):
        segment = words[i:i + WORDS_PER_CHUNK]
        if len(segment) < 75:        # drop the runt at the end of a book
            break
        text = " ".join(segment)
        chunks.append({
            "id":          f"{d['title'].replace(' ', '_')}#chunk{chunk_index}",
            "title":       d["title"],
            "text":        text,
            "preview":     text[:400],
            "source_path": d["path"],
            "chunk_index": chunk_index,
        })
        chunk_index += 1

print(f"{len(chunks)} chunks from {len(docs)} books")
print(f"average {np.mean([len(c['text'].split()) for c in chunks]):.0f} words per chunk")

8509 chunks from 13 books
average 300 words per chunk
time: 201 ms (started: 2026-08-29 21:53:11 +05:30)


In [11]:
chunks[0]

{'id': 'Moby-Dick#chunk0',
 'title': 'Moby-Dick',
 'text': 'MOBY-DICK; or, THE WHALE. By Herman Melville CONTENTS ETYMOLOGY. EXTRACTS (Supplied by a Sub-Sub-Librarian). CHAPTER 1. Loomings. CHAPTER 2. The Carpet-Bag. CHAPTER 3. The Spouter-Inn. CHAPTER 4. The Counterpane. CHAPTER 5. Breakfast. CHAPTER 6. The Street. CHAPTER 7. The Chapel. CHAPTER 8. The Pulpit. CHAPTER 9. The Sermon. CHAPTER 10. A Bosom Friend. CHAPTER 11. Nightgown. CHAPTER 12. Biographical. CHAPTER 13. Wheelbarrow. CHAPTER 14. Nantucket. CHAPTER 15. Chowder. CHAPTER 16. The Ship. CHAPTER 17. The Ramadan. CHAPTER 18. His Mark. CHAPTER 19. The Prophet. CHAPTER 20. All Astir. CHAPTER 21. Going Aboard. CHAPTER 22. Merry Christmas. CHAPTER 23. The Lee Shore. CHAPTER 24. The Advocate. CHAPTER 25. Postscript. CHAPTER 26. Knights and Squires. CHAPTER 27. Knights and Squires. CHAPTER 28. Ahab. CHAPTER 29. Enter Ahab; to Him, Stubb. CHAPTER 30. The Pipe. CHAPTER 31. Queen Mab. CHAPTER 32. Cetology. CHAPTER 33. The Specksnyder.

time: 1.05 ms (started: 2026-08-29 21:53:26 +05:30)


In [12]:
import time
from concurrent.futures import ThreadPoolExecutor

MAX_WORKERS = 6
RETRIES     = 3


def embed_text(text: str) -> np.ndarray:
    """Embed one string, unit-normalised, with retries. Used everywhere below."""
    for attempt in range(RETRIES):
        try:
            v = np.asarray(
                client.embeddings(model=EMBED_MODEL, prompt=text)["embedding"],
                dtype="float32",
            )
            return v / (np.linalg.norm(v) + 1e-12)
        except Exception:
            if attempt == RETRIES - 1:
                raise
            time.sleep(0.6 * (attempt + 1))

time: 392 µs (started: 2026-08-29 21:53:43 +05:30)


In [13]:
# ~5-6 minutes for 8,509 chunks. Good moment for a break.
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    emb_vectors = list(tqdm(
        pool.map(lambda c: embed_text(f'title: {c["title"]} | text: {c["text"]}'), chunks),
        total=len(chunks),
        desc=f"embedding ({EMBED_MODEL})",
    ))

emb = np.vstack(emb_vectors)
print("embeddings:", emb.shape)

embedding (embeddinggemma:latest): 100%|██████████| 8509/8509 [04:01<00:00, 35.18it/s]

embeddings: (8509, 768)
time: 4min 1s (started: 2026-08-29 21:53:49 +05:30)


In [14]:
import faiss

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("FAISS index:", index.ntotal, "vectors of dim", emb.shape[1])

FAISS index: 8509 vectors of dim 768
time: 34 ms (started: 2026-08-29 21:57:51 +05:30)


In [15]:
query = "Best detective in the world"

q_vec = embed_text(f"task: search result | query: {query}")
D, I = index.search(q_vec.reshape(1, -1), TOPK)

print(f"Top {TOPK} for: {query!r}\n")
for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):
    c = chunks[idx]
    print(f"  #{rank}  {score:.3f}  {c['title']} (chunk {c['chunk_index']})")
    pretty_print("      ", c["preview"][:300])
    print()

Top 5 for: 'Best detective in the world'

  #1  0.413  Adventures of Sherlock Holmes (chunk 63)
       confidence in Mr. Holmes, sir,” said the police agent loftily. “He has
his own little methods, which are, if he won’t mind my saying so, just a little
too theoretical and fantastic, but he has the makings of a detective in him. It
is not too much to say that once or twice, as in that business of the

  #2  0.410  Adventures of Sherlock Holmes (chunk 132)
       corresponds with the injuries. There is no sign of any other weapon.”
“And the murderer?” “Is a tall man, left-handed, limps with the right leg, wears
thick-soled shooting-boots and a grey cloak, smokes Indian cigars, uses a cigar-
holder, and carries a blunt pen-knife in his pocket. There are severa

  #3  0.400  Adventures of Sherlock Holmes (chunk 134)
       a strong presumption that the person whom McCarthy expected to meet him
at Boscombe Pool was someone who had been in Australia.” “What of the rat,
then?” Sherlock Holme

In [16]:
KEY_TO_IDX = {(c["source_path"], c["chunk_index"]): i for i, c in enumerate(chunks)}
print(f"{len(KEY_TO_IDX)} (book, chunk) keys")

8509 (book, chunk) keys
time: 1.59 ms (started: 2026-08-29 22:05:30 +05:30)


In [17]:
def expand_with_neighbors_simple(I, chunks, neighbors=1, max_out=8):
    """
    For each FAISS hit in I[0], emit the hit plus its ±neighbors chunks from the
    same book, in reading order. Deduplicated (two hits can share a neighbour),
    hit order preserved, capped at max_out chunks.

    Returns a flat list — one dict per chunk, so each becomes one numbered,
    citable passage in the prompt.
    """
    seen, contexts = set(), []

    for gi in I[0]:
        c = chunks[int(gi)]
        for delta in range(-neighbors, neighbors + 1):
            j = KEY_TO_IDX.get((c["source_path"], c["chunk_index"] + delta))
            if j is None or j in seen:
                continue
            seen.add(j)
            n = chunks[j]
            contexts.append({
                "title":        n["title"],
                "source_path":  n["source_path"],
                "chunk_index":  n["chunk_index"],
                "text":         n["text"],
                "approx_words": len(n["text"].split()),
                "is_hit":       delta == 0,
            })
            if len(contexts) >= max_out:
                return contexts
    return contexts

time: 607 µs (started: 2026-08-29 22:07:02 +05:30)


In [19]:
def search_windowed(query: str, topk: int = 3, neighbors: int = 1, max_out: int = 12):
    """
    Embed the query -> FAISS top-k -> expand each hit with ±neighbours.
    Returns (df_hits, contexts): the raw hits for you, the contexts for the LLM.
    """
    q_vec = embed_text(f"task: search result | query: {query}")
    D, I = index.search(q_vec.reshape(1, -1), max(topk, TOPK))
 # Chunk 5, 8 and 23
    contexts = expand_with_neighbors_simple(
        np.array([I[0][:topk]]), chunks, neighbors=neighbors, max_out=max_out
    )

    df_hits = pd.DataFrame([{
        "rank":        rank,
        "score":       round(float(score), 3),
        "title":       chunks[idx]["title"],
        "chunk_index": chunks[idx]["chunk_index"],
    } for rank, (score, idx) in enumerate(zip(D[0].tolist(), I[0].tolist()), start=1)])

    return df_hits, contexts

time: 782 µs (started: 2026-08-29 22:08:04 +05:30)


In [21]:
QUESTION = "Who is the best detective in the world?"

time: 147 µs (started: 2026-08-29 22:08:33 +05:30)


In [22]:
df_hits, contexts = search_windowed(QUESTION, topk=3, neighbors=1)

print("=== raw FAISS hits ===")
display(df_hits)

print(f"\n=== expanded context: {len(contexts)} chunks ===")
display(pd.DataFrame([{
    "title":       c["title"],
    "chunk_index": c["chunk_index"],
    "is_hit":      c["is_hit"],
    "words":       c["approx_words"],
    "names_the_answer": "swamp adder" in c["text"].lower(),
} for c in contexts]))

=== raw FAISS hits ===


,rank,score,title,chunk_index
0,1,0.428,Adventures of Sherlock Holmes,63
1,2,0.419,Adventures of Sherlock Holmes,132
2,3,0.411,Adventures of Sherlock Holmes,5
3,4,0.408,Adventures of Sherlock Holmes,134
4,5,0.404,Adventures of Sherlock Holmes,109



=== expanded context: 9 chunks ===


,title,chunk_index,is_hit,words,names_the_answer
0,Adventures of Sherlock Holmes,62,False,300,False
1,Adventures of Sherlock Holmes,63,True,300,False
2,Adventures of Sherlock Holmes,64,False,300,False
3,Adventures of Sherlock Holmes,131,False,300,False
4,Adventures of Sherlock Holmes,132,True,300,False
5,Adventures of Sherlock Holmes,133,False,300,False
6,Adventures of Sherlock Holmes,4,False,300,False
7,Adventures of Sherlock Holmes,5,True,300,False
8,Adventures of Sherlock Holmes,6,False,300,False


time: 598 ms (started: 2026-08-29 22:08:46 +05:30)


In [27]:
def answer(question: str, contexts: list, model: str = GEN_MODEL, **kw) -> str:
    """Ask `model` to answer `question` using only `contexts`, with citations."""
    block = "\n\n".join(
        f"[{i}] ({c['title']}, chunk {c['chunk_index']})\n{c['text']}"
        for i, c in enumerate(contexts, start=1)
    )
    resp = client.chat(
        model=model,
        messages=[{"role": "user", "content":
            "Answer the question using ONLY the passages below. Cite the passage numbers "
            "you used, like [2].\n"
            "The passages are excerpts from novels, so the speaker of a line may be named "
            "in a neighbouring passage rather than the one containing the line — read them "
            "together before deciding who is speaking.\n"
            "If the passages genuinely do not answer the question, say so and state what "
            "they do show instead.\n\n"
            f"{block}\n\nQuestion: {question}\nAnswer:"
        }],
        options={"temperature": 0.0},
        **kw,
    )
    return resp["message"]["content"].strip()

time: 786 µs (started: 2026-08-29 22:23:58 +05:30)


In [25]:
_, ctx_c = search_windowed(QUESTION, topk=3, neighbors=1)
print(f"context: {len(ctx_c)} chunks, {sum(c['approx_words'] for c in ctx_c)} words\n")
pretty_print(answer(QUESTION, ctx_c))

context: 9 chunks, 2700 words

This question cannot be answered based on the provided passages. The passages do
not directly compare the detective skills of different individuals, nor do they
provide a clear ranking of detectives. However, they do portray Sherlock Holmes
as a skilled and renowned detective, with abilities that are described as
"remarkable" and "theoretical and fantastic" (Passage 2).
time: 10.2 s (started: 2026-08-29 22:16:07 +05:30)


In [28]:
pretty_print("qwen3.5:9b →", answer(QUESTION, ctx_c, model="qwen3.5:9b", think=True))

qwen3.5:9b → The provided passages do not answer the question of who is the best
detective in the world. They show that Sherlock Holmes is a detective who has
been "more nearly correct than the official force" on occasion [2], and Watson
acknowledges that his eyes are as good as Holmes's own [8].
time: 2min 23s (started: 2026-08-29 22:24:00 +05:30)


In [ ]:
import json

Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

with open(Path(ARTIFACTS_DIR) / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False)

np.save(Path(ARTIFACTS_DIR) / "embeddings.npy", emb)
faiss.write_index(index, str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))

with open(Path(ARTIFACTS_DIR) / "config.json", "w") as f:
    json.dump({
        "EMBED_MODEL":     EMBED_MODEL,
        "GEN_MODEL":       GEN_MODEL,
        "WORDS_PER_CHUNK": WORDS_PER_CHUNK,
        "OVERLAP_WORDS":   OVERLAP_WORDS,
        "TOPK":            TOPK,
    }, f, indent=2)

for f in sorted(Path(ARTIFACTS_DIR).iterdir()):
    print(f"  {f.name:20} {f.stat().st_size/1e6:8.2f} MB")

what if RAG retrives the incorrect information does LLM correct it?


In [ ]:
I have data stored in my company’s database, say PostgreSQL. To retrieve and respond to questions based on this data, we can use a DB MCP server as a tool. In which scenarios is RAG actually needed? For the PDF use case we discussed, could we store the extracted PDF content in a normal database (rather than a vector database) and use DB MCP to retrieve the relevant information? Is my understanding correct, or are there cases where RAG/vector search would still be required?
